In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [2]:
nvdia_rag = read_repo_data('NVIDIA-AI-Blueprints', 'rag')

In [3]:
print(f"nvdia rag documents: {len(nvdia_rag)}")


nvdia rag documents: 109


In [4]:
print(f"nvdia rag document1 content: {nvdia_rag[1]}")

nvdia rag document1 content: {'content': '# NVIDIA RAG Blueprint\n\nReference implementation for a Retrieval Augmented Generation pipeline. Python 3.11+ backend (FastAPI + LangChain), React/TypeScript frontend, deployable via Docker Compose or Helm.\n\n## Project structure\n\n```\nsrc/nvidia_rag/\n├── rag_server/        # RAG query/response server (FastAPI)\n├── ingestor_server/   # Document ingestion server (FastAPI)\n└── utils/             # Shared utilities\nfrontend/              # React + TypeScript UI (pnpm)\ndeploy/\n├── compose/           # Docker Compose files and env configs\n└── helm/              # Helm charts (standard + MIG-slicing)\ndocs/                  # User-facing documentation (Sphinx, RST/MD)\ntests/\n├── unit/              # No network calls allowed\n└── integration/       # Network calls permitted\nnotebooks/             # Jupyter notebooks for evaluation and examples\n```\n\n## Development commands\n\n### Backend (Python)\n\n```bash\nuv sync                    